In [ ]:
# 다중 상속과 메서드 결정 순서 (p.567)

# 다중 상속을 구현하는 언어는 슈퍼클래스들이 동일한 이름으로 메서드를 구현할 때
# 발생하는 이름 충돌 문제를 해결해야 한다


# 다이아몬드 문제 : 이름 충돌 문제

In [3]:
class Root:           # 가장 최상위 부모 클래스
    def ping(self):
        print(f'{self}.ping() in Root')
        print("-" * 30)

    def pong(self):
        print(f'{self}.pong() in Root')
        print("-" * 30)

    def __repr__(self):     
        cls_name = type(self).__name__
        return f'<instance of {cls_name}>'     # 객체를 출력(print)할 때 보기 좋게 표현하는 함수

class A(Root):      # Root를 상속받는 첫 번째 자식 클래스
    def ping(self):
        print(f'{self}.ping() in A')
        print("-" * 30)
        super().ping()      

    def pong(self):
        print(f'{self}.pong() in A')
        print("-" * 30)
        super().pong()

class B(Root):
    def ping(self):
        print(f'{self}.ping() in B')
        print("-" * 30)
        super().ping()

    def pong(self):
        print(f'{self}.pong in B')
        print("-" * 30)

class Leaf(A, B):
    def ping(self):
        print(f'{self}.ping() in Leaf')
        print("-" * 30)
        super().ping()

In [4]:
leaf1 = Leaf()
leaf1.ping()

<instance of Leaf>.ping() in Leaf
------------------------------
<instance of Leaf>.ping() in A
------------------------------
<instance of Leaf>.ping() in B
------------------------------
<instance of Leaf>.ping() in Root
------------------------------


In [ ]:
# leaf1.ping()이 호출됨으로 인하여 A와 B로 흐름이 분기되고 
# A, B각각에서 Root가 한번씩 호출되면 결국 결과는 "ping() in Root"가 두 번 출력되어야 할 것 같다. 
# 그런데 실행 결과는 한번 뿐인가?

### 파이썬의 다중 상속은 트리를 따라 재귀적으로 내려가는 방식이 아니라, MRO라는 하나의 선형(Linear) 순서를 따라 이동합니다.
### 우리가 흔히 생각하는 방식 (X)

          Root
         /    \
        A      B
         \    /
         Leaf

### 그리고 Leaf.ping()을 호출하면

Leaf
 ├── A
 │     └── Root
 │
 └── B
       └── Root


### 이렇게 분기(branch) 된다고 생각합니다. 이렇게 동작한다면,
Root.ping()
### 이 두 번 호출되는 것이 맞습니다.

In [7]:
# 파이썬은 상속 트리를 탐색하지 않는다
# 대신 MRO를 먼저 계산한다.

print(Leaf.__mro__)

(<class '__main__.Leaf'>, <class '__main__.A'>, <class '__main__.B'>, <class '__main__.Root'>, <class 'object'>)


In [ ]:
# 이미 하나의 직선으로 만들어 놓는다.
# 그래서 호출도 한 줄로 진행된다.

## 만일 Root가 두 번 호출된다면?
### 이 문제를 "Diamond Problem(다이아몬드 문제) 라고 부른다."

### 파이썬은 "모든 클래스는 한 번만 실행한다"라는 규칙을 사용한다.

In [ ]:
'''
처음에 생각한 방식은 "상속 그래프를 따라 분기 탐색"하는 방식입니다.
하지만 파이썬의 super()는 상속 그래프를 따라 내려가지 않고, 
MRO라는 '한 줄짜리 호출 목록'을 따라 이동합니다. 
따라서 Root는 MRO에 한 번만 등장하므로 Root.ping()도 정확히 한 번만 실행됩니다.
이 개념을 이해하면 다중 상속의 동작 원리를 거의 완전히 이해한 것이라고 볼 수 있습니다.
'''
# MRO : Method Resolution Order : 메서드 탐색 순서
# 왜 필요한가? - 단일 상속에는 문제가 없다.
# 그러나 다중 상속에서는 문제가 발생한다. -> 메서드를 이 순서대로 찾아라 라고 규칙을 만든다.


In [ ]:
# super()도 MRO를 따른다.

''' 
부모 클래스를 호출한다 라고 배우지만,

정확히는 MRO에서 다음 순서의 클래스를 호출한다 입니다.
'''

'''
MRO를 한 문장으로 기억하기

초보자에게 가장 이해하기 쉬운 정의는 다음과 같습니다.

MRO(Method Resolution Order)는 파이썬이 메서드를 찾거나 super()가 다음에 호출할 클래스를 결정하기 위해 사용하는 '우선순위 목록'이다.

이 목록은 상속 구조를 그대로 따라가는 것이 아니라, C3 Linearization이라는 알고리즘으로 계산됩니다. 덕분에 다중 상속에서도 각 클래스가 한 번씩만 방문되며, 앞에서 살펴본 **다이아몬드 문제(Diamond Problem)**를 깔끔하게 해결할 수 있습니다.

'''